In [1]:
import pandas as pd

def write_degrees(df, name):
    in_deg = df.groupby('target neuron id').size().rename('in_deg')
    out_deg = df.groupby('source neuron id').size().rename('out_deg')
    degrees = pd.concat([in_deg, out_deg], axis=1).fillna(0).reset_index().astype(int).sort_values(by=['in_deg', 'out_deg'], ascending=False)
    degrees['dir_ratio'] = degrees.apply(lambda x: round((x['in_deg']) / (x['in_deg'] + x['out_deg']), ndigits=3), axis=1)
    degrees['total_deg'] = degrees.apply(lambda x: (x['in_deg'] + x['out_deg']), axis=1)
    degrees['rank_pct'] = degrees['total_deg'].rank(pct=True).round(3)
    degrees.to_csv(f"degrees_{name}.csv", index=False)
    print(degrees['index'][0:5].to_list())
    return degrees

maol = pd.read_csv("maol_1.1_edge_list.csv")
maol_degrees = write_degrees(maol, "maol")
print(maol_degrees.shape)

mcns = pd.read_csv("mcns_0.9_edge_list.csv")
mcns_degrees = write_degrees(mcns, "mcns")
print(mcns_degrees.shape)

# manc = pd.read_csv("manc_1.2.1_edge_list.csv")
# manc_degrees = write_degrees(manc, "manc")
# print(manc_degrees.shape)

# banc = pd.read_csv("banc_626_edge_list.csv")
# banc_degrees = write_degrees(banc, "banc")
# print(banc_degrees.shape)

fafb = pd.read_csv("fafb_783_edge_list.csv")
fafb_degrees = write_degrees(fafb, "fafb")
print(fafb_degrees.shape)

[10009, 10051, 10330, 10302, 10093]
(51669, 6)
[10009, 10157, 10085, 10093, 537032]
(165820, 6)
[720575940626979621, 720575940628908548, 720575940613583001, 720575940624547622, 720575940620975696]
(138584, 6)


In [2]:
result = (
    maol_degrees
    .merge(fafb_degrees, on=['dir_ratio', 'rank_pct'])
    .rename(columns={'index_x': 'maol_id', 'index_y': 'fafb_id'})
    .merge(mcns_degrees, on=['dir_ratio', 'rank_pct'])
    .rename(columns={'index': 'mcns_id'})
    .sort_values(by=['dir_ratio'], ascending=False)
)

result.to_csv("dir_candidates.csv")

maol_ids = set(result['maol_id'])
fafb_ids = set(result['fafb_id'])
mcns_ids = set(result['mcns_id'])

# Find edges where both endpoints are candidate neurons
maol_internal = maol[(maol['source neuron id'].isin(maol_ids)) & 
                     (maol['target neuron id'].isin(maol_ids))]
fafb_internal = fafb[(fafb['source neuron id'].isin(fafb_ids)) & 
                     (fafb['target neuron id'].isin(fafb_ids))]
mcns_internal = mcns[(mcns['source neuron id'].isin(mcns_ids)) & 
                     (mcns['target neuron id'].isin(mcns_ids))]

maol_connected = set(maol_internal['source neuron id']).union(set(maol_internal['target neuron id']))
fafb_connected = set(fafb_internal['source neuron id']).union(set(fafb_internal['target neuron id']))
mcns_connected = set(mcns_internal['source neuron id']).union(set(mcns_internal['target neuron id']))

filtered_results = result[
    result['maol_id'].isin(maol_connected) & 
    result['fafb_id'].isin(fafb_connected) & 
    result['mcns_id'].isin(mcns_connected)
]

# # 3. Find the maximum difference among the three columns
# filtered_results['dir_ratio_diff'] = (
#     filtered_results[['maol_dir_ratio', 'fafb_dir_ratio', 'mcns_dir_ratio']].max(axis=1) - 
#     filtered_results[['maol_dir_ratio', 'fafb_dir_ratio', 'mcns_dir_ratio']].min(axis=1)
# )

# filtered_results['rank_percentile_diff'] = (
#     filtered_results[['maol_rank_pct', 'fafb_rank_pct', 'mcns_rank_pct']].max(axis=1) - 
#     filtered_results[['maol_rank_pct', 'fafb_rank_pct', 'mcns_rank_pct']].min(axis=1)
# )

# filtered_results = filtered_results[
#     (filtered_results['dir_ratio_diff'] < 0.005) &        # Very tight directional ratio match
#     (filtered_results['rank_percentile_diff'] < 0.02)     # Highly identical degree rank ranking
# ]
print(filtered_results.shape)

(1882, 14)


In [3]:
import networkx as nx
from itertools import combinations
from networkx.algorithms.clique import find_cliques

In [25]:
strict_candidates = filtered_results[filtered_results['rank_pct'] > 0]

# 1. Prepare your triplets
initial_triplets = list(zip(strict_candidates['maol_id'], strict_candidates['fafb_id'], strict_candidates['mcns_id']))
print(f"Candidate pool contains {len(initial_triplets)} triplets.")

# 2. Precompute edge sets as fast hash lookups
maol_edges = set(zip(maol['source neuron id'], maol['target neuron id']))
fafb_edges = set(zip(fafb['source neuron id'], fafb['target neuron id']))
mcns_edges = set(zip(mcns['source neuron id'], mcns['target neuron id']))

active_triplet_indices = set()
for i, j in combinations(range(len(initial_triplets)), 2):
    m_i, f_i, c_i = initial_triplets[i]
    m_j, f_j, c_j = initial_triplets[j]
    
    if (m_i == m_j) or (f_i == f_j) or (c_i == c_j):
        continue
        
    if (m_i, m_j) in maol_edges and (f_i, f_j) in fafb_edges and (c_i, c_j) in mcns_edges:
        active_triplet_indices.add(i)
        active_triplet_indices.add(j)
    if (m_j, m_i) in maol_edges and (f_j, f_i) in fafb_edges and (c_j, c_i) in mcns_edges:
        active_triplet_indices.add(i)
        active_triplet_indices.add(j)

triplets = [initial_triplets[idx] for idx in sorted(active_triplet_indices)]
print(f"Interconnected candidate pool size: {len(triplets)}")

# 3. Build the Compatibility Graph
compatibility_graph = nx.Graph()
compatibility_graph.add_nodes_from(range(len(triplets)))

print("Building compatibility graph...")
for i, j in combinations(range(len(triplets)), 2):
    m_i, f_i, c_i = triplets[i]
    m_j, f_j, c_j = triplets[j]

    # Constraint 1: 1-to-1 Node Mapping
    if (m_i == m_j) or (f_i == f_j) or (c_i == c_j):
        continue
        
    # Constraint 2: Structural Isomorphism (Forward Direction i -> j)
    m_ij = (m_i, m_j) in maol_edges
    f_ij = (f_i, f_j) in fafb_edges
    c_ij = (c_i, c_j) in mcns_edges
    if not (m_ij == f_ij == c_ij):
        continue

    # Constraint 2: Structural Isomorphism (Reverse Direction j -> i)
    m_ji = (m_j, m_i) in maol_edges
    f_ji = (f_j, f_i) in fafb_edges
    c_ji = (c_j, c_i) in mcns_edges
    if not (m_ji == f_ji == c_ji):
        continue

    # If it survives all checks, they are mutually compatible!
    compatibility_graph.add_edge(i, j)

edge_count = compatibility_graph.number_of_edges()
node_count = compatibility_graph.number_of_nodes()
max_edges = (node_count * (node_count - 1))/2
print(f"Compatibility Graph Complete: {node_count} nodes, {edge_count} edges ({(edge_count/max_edges)*100}% density).")

# 4. Extract the Maximum Clique directly using Branch-and-Bound
print("Computing the single maximum clique (bypassing all maximal cliques)...")
# weight=None treats all nodes as weight 1, finding the true maximum cardinality clique
largest_clique_nodes, clique_size = nx.max_weight_clique(compatibility_graph, weight=None)

# Map the node indices back to your actual neuron IDs
final_alignment = [triplets[idx] for idx in largest_clique_nodes]

print(f"\nSUCCESS! Found a perfectly matched subgraph alignment of size: {clique_size}")
print("Sample aligned triplets (MAOL, FAFB, MCNS):")
print(final_alignment[:5])

Candidate pool contains 1882 triplets.
Interconnected candidate pool size: 0
Building compatibility graph...


ZeroDivisionError: division by zero

In [12]:
# Assuming 'final_alignment' is your list of matched tuples: [(maol_id, fafb_id, mcns_id), ...]

# 1. Convert the list of triplets into a DataFrame in a single bulk operation
alignment_df = pd.DataFrame(
    final_alignment, 
    columns=['maol_id', 'fafb_id', 'mcns_id']
)

# 2. Save to a CSV file
# index=False prevents pandas from adding an unnecessary row-number column to your file
output_filename = "final_circuit_alignment.csv"
alignment_df.to_csv(output_filename, index=False)

print(f"Successfully saved {len(alignment_df)} aligned triplets to '{output_filename}'!")

Successfully saved 44 aligned triplets to 'final_circuit_alignment.csv'!


In [13]:
import pandas as pd
from itertools import combinations

def verify_alignment(alignment_path, maol_path, fafb_path, mcns_path):
    print("="*60)
    print("RUNNING NEURAL CIRCUIT ALIGNMENT AUDIT")
    print("="*60)
    
    # 1. Load all data
    print("Loading data files...")
    alignment_df = pd.read_csv(alignment_path)
    maol = pd.read_csv(maol_path)
    fafb = pd.read_csv(fafb_path)
    mcns = pd.read_csv(mcns_path)
    
    total_triplets = len(alignment_df)
    print(f"Loaded {total_triplets} aligned triplets for evaluation.\n")
    
    # --- AUDIT STEP 1: 1-to-1 Node Mapping Uniqueness ---
    print("Audit 1: Checking 1-to-1 Node Mapping Uniqueness...")
    is_valid_mapping = True
    cols_to_check = ['maol_id', 'fafb_id', 'mcns_id']
    
    for col in cols_to_check:
        unique_count = alignment_df[col].nunique()
        if unique_count != total_triplets:
            print(f"❌ FAILURE: Duplicate IDs detected in '{col}'! ({unique_count} unique IDs out of {total_triplets} rows)")
            is_valid_mapping = False
            
    if is_valid_mapping:
        print("✅ SUCCESS: 1-to-1 node constraints hold. All neuron mappings are unique.")
    
    # --- AUDIT STEP 2: Induced Structural Isomorphism ---
    print("\nAudit 2: Building global edge hash maps for lookups...")
    maol_edges = set(zip(maol['source neuron id'], maol['target neuron id']))
    fafb_edges = set(zip(fafb['source neuron id'], fafb['target neuron id']))
    mcns_edges = set(zip(mcns['source neuron id'], mcns['target neuron id']))
    
    triplets = list(zip(alignment_df['maol_id'], alignment_df['fafb_id'], alignment_df['mcns_id']))
    
    print("Auditing structural isomorphism across all distinct pairs...")
    structural_errors = 0
    total_pairs_checked = 0
    
    for i, j in combinations(range(len(triplets)), 2):
        total_pairs_checked += 1
        m_i, f_i, c_i = triplets[i]
        m_j, f_j, c_j = triplets[j]
        
        # Check Forward Directional Edge (i -> j)
        m_ij = (m_i, m_j) in maol_edges
        f_ij = (f_i, f_j) in fafb_edges
        c_ij = (c_i, c_j) in mcns_edges
        
        if not (m_ij == f_ij == c_ij):
            structural_errors += 1
            if structural_errors <= 3:  # Prevent flooding the terminal with errors
                print(f"  ❌ Mismatch (Forward i -> j) between Row {i} and Row {j}:")
                print(f"     MAOL ({m_i} -> {m_j}): {m_ij}")
                print(f"     FAFB ({f_i} -> {f_j}): {f_ij}")
                print(f"     MCNS ({c_i} -> {c_j}): {c_ij}")
                
        # Check Reverse Directional Edge (j -> i)
        m_ji = (m_j, m_i) in maol_edges
        f_ji = (f_j, f_i) in fafb_edges
        c_ji = (c_j, c_i) in mcns_edges
        
        if not (m_ji == f_ji == c_ji):
            structural_errors += 1
            if structural_errors <= 3:
                print(f"  ❌ Mismatch (Reverse j -> i) between Row {i} and Row {j}:")
                print(f"     MAOL ({m_j} -> {m_i}): {m_ji}")
                print(f"     FAFB ({f_j} -> {f_i}): {f_ji}")
                print(f"     MCNS ({c_j} -> {c_i}): {c_ji}")

    if structural_errors > 3:
        print(f"  [... Suppressed {structural_errors - 3} additional edge mismatch printouts ...]")

    print(f"Evaluated {total_pairs_checked} unique triplet combinations.")
    if structural_errors == 0:
        print("✅ SUCCESS: Structural isomorphism verified. Induced directed subgraphs are identical mirrors!")
    else:
        print(f"❌ FAILURE: Found {structural_errors} contradictory connection states.")

    # --- FINAL VERDICT ---
    print("\n" + "="*60)
    if is_valid_mapping and structural_errors == 0:
        print("🎉 AUDIT PASSED: The alignment is mathematically 100% valid! 🎉")
    else:
        print("🚨 AUDIT FAILED: The output contains violations. Review logic rules. 🚨")
    print("="*60)

# Run verification script
verify_alignment(
    alignment_path="final_circuit_alignment.csv",
    maol_path="maol_1.1_edge_list.csv",
    fafb_path="fafb_783_edge_list.csv",
    mcns_path="mcns_0.9_edge_list.csv"
)

RUNNING NEURAL CIRCUIT ALIGNMENT AUDIT
Loading data files...
Loaded 44 aligned triplets for evaluation.

Audit 1: Checking 1-to-1 Node Mapping Uniqueness...
✅ SUCCESS: 1-to-1 node constraints hold. All neuron mappings are unique.

Audit 2: Building global edge hash maps for lookups...
Auditing structural isomorphism across all distinct pairs...
Evaluated 946 unique triplet combinations.
✅ SUCCESS: Structural isomorphism verified. Induced directed subgraphs are identical mirrors!

🎉 AUDIT PASSED: The alignment is mathematically 100% valid! 🎉


In [46]:
import pandas as pd
import networkx as nx
from itertools import combinations
import numpy as np

print("Step 1: Filtering for top-tier Circuit Hubs via Quantiles...")

# 🌟 THE FIX: Grab only the top 2% or 5% of the most connected hubs in each dataset
# Adjust 0.95 (top 5%) or 0.98 (top 2%) based on how selective you want to be
QUANTILE_THRESHOLD = 0.6  

maol_cutoff = maol_degrees['total_deg'].quantile(QUANTILE_THRESHOLD, QUANTILE_THRESHOLD + 0.02).values
fafb_cutoff = fafb_degrees['total_deg'].quantile(QUANTILE_THRESHOLD, QUANTILE_THRESHOLD + 0.02).values
mcns_cutoff = mcns_degrees['total_deg'].quantile(QUANTILE_THRESHOLD, QUANTILE_THRESHOLD + 0.02).values

print(f"Calculated Hub Cutoffs -> MAOL: {maol_cutoff:.1f}, FAFB: {fafb_cutoff:.1f}, MCNS: {mcns_cutoff:.1f}")

maol_hubs = maol_degrees[maol_degrees['total_deg'] >= maol_cutoff].reset_index().rename(columns={'index': 'neuron_id'})
fafb_hubs = fafb_degrees[fafb_degrees['total_deg'] >= fafb_cutoff].reset_index().rename(columns={'index': 'neuron_id'})
mcns_hubs = mcns_degrees[mcns_degrees['total_deg'] >= mcns_cutoff].reset_index().rename(columns={'index': 'neuron_id'})

print(f"Safe Hub counts -> MAOL: {len(maol_hubs)}, FAFB: {len(fafb_hubs)}, MCNS: {len(mcns_hubs)}")

print("\nStep 2: Generating Candidate Triplets via Proximity Tolerance...")
# Instead of a rigid pandas merge, we find entries close to each other in fingerprint space
DIR_TOLERANCE = 0.005
RANK_TOLERANCE = 0.005

initial_triplets = []

# High-performance proximity matching using numpy broadcasting
maol_dir = maol_hubs['dir_ratio'].values
maol_rank = maol_hubs['rank_pct'].values

for _, f_row in fafb_hubs.iterrows():
    # Find MAOL matches for this FAFB neuron
    dir_match = np.abs(maol_dir - f_row['dir_ratio']) <= DIR_TOLERANCE
    rank_match = np.abs(maol_rank - f_row['rank_pct']) <= RANK_TOLERANCE
    valid_maol_indices = np.where(dir_match & rank_match)[0]
    
    if len(valid_maol_indices) == 0:
        continue
        
    # Find MCNS matches for this FAFB neuron
    mcns_matches = mcns_hubs[
        (np.abs(mcns_hubs['dir_ratio'] - f_row['dir_ratio']) <= DIR_TOLERANCE) &
        (np.abs(mcns_hubs['rank_pct'] - f_row['rank_pct']) <= RANK_TOLERANCE)
    ]
    
    # Create the triplets
    for m_idx in valid_maol_indices:
        m_id = maol_hubs.iloc[m_idx]['neuron_id']
        for _, c_row in mcns_matches.iterrows():
            initial_triplets.append((int(m_id), int(f_row['neuron_id']), int(c_row['neuron_id'])))

print()
print(f"Generated {len(initial_triplets)} high-fidelity candidate triplets.")

print("\nStep 3: Building Global Edge Sets...")
maol_edges = set(zip(maol['source neuron id'], maol['target neuron id']))
fafb_edges = set(zip(fafb['source neuron id'], fafb['target neuron id']))
mcns_edges = set(zip(mcns['source neuron id'], mcns['target neuron id']))

print("Step 4: Pruning via Mutual Biological Edges...")
active_triplet_indices = set()

for i, j in combinations(range(len(initial_triplets)), 2):
    m_i, f_i, c_i = initial_triplets[i]
    m_j, f_j, c_j = initial_triplets[j]

    # Skip 1-to-1 violations during edge screening
    if (m_i == m_j) or (f_i == f_j) or (c_i == c_j):
        continue
        
    # Forward biological edge must exist simultaneously across all three
    if (m_i, m_j) in maol_edges and (f_i, f_j) in fafb_edges and (c_i, c_j) in mcns_edges:
        active_triplet_indices.add(i)
        active_triplet_indices.add(j)

    # Reverse biological edge must exist simultaneously across all three
    if (m_j, m_i) in maol_edges and (f_j, f_i) in fafb_edges and (c_j, c_i) in mcns_edges:
        active_triplet_indices.add(i)
        active_triplet_indices.add(j)

triplets = [initial_triplets[idx] for idx in sorted(active_triplet_indices)]
print(f"🎉 Success! Active, interconnected candidate pool size: {len(triplets)}")

# 5. Build the Compatibility Graph and extract the Clique
if len(triplets) > 0:
    print("\nStep 5: Extracting Perfectly Isomorphic Subgraph...")
    compatibility_graph = nx.Graph()
    compatibility_graph.add_nodes_from(range(len(triplets)))

    for i, j in combinations(range(len(triplets)), 2):
        m_i, f_i, c_i = triplets[i]
        m_j, f_j, c_j = triplets[j]

        if (m_i == m_j) or (f_i == f_j) or (c_i == c_j):
            continue
            
        # Perfect structural isomorphism check
        if not ((m_i, m_j) in maol_edges == (f_i, f_j) in fafb_edges == (c_i, c_j) in mcns_edges):
            continue
        if not ((m_j, m_i) in maol_edges == (f_j, f_i) in fafb_edges == (c_j, c_i) in mcns_edges):
            continue

        compatibility_graph.add_edge(i, j)

    largest_clique_nodes, clique_size = nx.max_weight_clique(compatibility_graph, weight=None)
    final_alignment = [triplets[idx] for idx in largest_clique_nodes]
    
    print(f"\n✨ FINAL CIRCUIT FOUND! Size: {len(final_alignment)} aligned neurons.")
    
    alignment_df = pd.DataFrame(final_alignment, columns=['maol_id', 'fafb_id', 'mcns_id'])
    alignment_df.to_csv("final_circuit_alignment.csv", index=False)
else:
    print("\n❌ No active triplets found. Try increasing DIR_TOLERANCE or lowering MIN_DEGREE.")

Step 1: Filtering for top-tier Circuit Hubs via Quantiles...


ValueError: 0.62 is not a valid method. Use one of: dict_keys(['inverted_cdf', 'averaged_inverted_cdf', 'closest_observation', 'interpolated_inverted_cdf', 'hazen', 'weibull', 'linear', 'median_unbiased', 'normal_unbiased', 'lower', 'higher', 'midpoint', 'nearest'])

In [19]:
import gravis as gv
import networkx as nx
import pandas as pd

network = pd.read_csv("final_circuit_alignment.csv")
maol_edges = pd.read_csv("maol_1.1_edge_list.csv")
maol_edges = maol_edges[(maol_edges['source neuron id'].isin(network['maol_id'])) & (maol_edges['target neuron id'].isin(network['maol_id']))]
print(len(maol_edges))

# graph = nx.from_pandas_edgelist(maol_edges, source='source neuron id', target='target neuron id', create_using=nx.DiGraph)
# gv.d3(graph.to_undirected())

0
